# Executive Hotel Booking — EDA Report
**Day 15 Assignment | End-to-End Exploratory Data Analysis**

This notebook follows the standard data science EDA workflow:
1. Data Understanding
2. Data Quality Assessment
3. Data Cleaning & Preprocessing
4. Univariate Analysis
5. Bivariate Analysis
6. Descriptive & Group-wise Statistics
7. Correlation Analysis
8. Visualizations (Matplotlib & Seaborn)
9. Key Business Insights
10. Management Recommendations

> Upload `Day15_Executive_Hotel_Booking_EDA_Dataset.csv` to your Colab session (or mount Google Drive) before running.


## 1. Setup & Data Loading

In [ ]:
# If needed, uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)


In [ ]:
# --- Load dataset ---
# Option A: file uploaded directly to Colab's working directory
FILE_PATH = "Day15_Executive_Hotel_Booking_EDA_Dataset.csv"

# Option B: upload via widget (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()
# FILE_PATH = list(uploaded.keys())[0]

df = pd.read_csv(FILE_PATH)
print("Shape:", df.shape)
df.head()


## 2. Understanding the Dataset

Basic structure, data types, and summary statistics of the raw data.

In [ ]:
df.info()


In [ ]:
print("Numeric summary:")
display(df.describe(include=[np.number]).T)

print("\nCategorical summary:")
display(df.describe(include=['object']).T)


In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())
print("Number of duplicate Booking_IDs:", df['Booking_ID'].duplicated().sum())


## 3. Data Quality Assessment

Checking missing values, incorrect data types, inconsistent categorical values, and outliers.

In [ ]:
# --- Missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_%': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_%', ascending=False)
missing_df


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=missing_df.index, y=missing_df['Missing_%'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('% Missing')
plt.title('Missing Values by Column')
plt.tight_layout()
plt.show()


In [ ]:
# --- Data type check ---
# Booking_Date, Arrival_Date, Reservation_Status_Date should be datetime, not object
for col in ['Booking_Date', 'Arrival_Date', 'Reservation_Status_Date']:
    print(col, '->', df[col].dtype)


In [ ]:
# --- Inconsistent / unexpected categorical values ---
categorical_cols = ['Hotel_Type', 'Hotel_Location', 'Country', 'Market_Segment',
                     'Distribution_Channel', 'Room_Type_Reserved', 'Room_Type_Assigned',
                     'Deposit_Type', 'Customer_Type', 'Meal_Type', 'Reservation_Status']

for col in categorical_cols:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts(dropna=False).head(10))
    print()


In [ ]:
# --- Logical consistency checks ---
print("Rows with 0 total guests (Adults+Children+Babies == 0):",
      ((df['Adults'] + df['Children'].fillna(0) + df['Babies']) == 0).sum())

print("Rows with 0 total nights:", (df['Total_Nights'] == 0).sum())

print("Rows where Total_Nights != Weekend_Nights + Weekday_Nights:",
      (df['Total_Nights'] != (df['Weekend_Nights'] + df['Weekday_Nights'])).sum())

print("Rows with negative ADR:", (df['ADR'] < 0).sum())
print("Rows with ADR == 0:", (df['ADR'] == 0).sum())


In [ ]:
# --- Outlier detection (IQR method) on key numeric columns ---
outlier_cols = ['Lead_Time_Days', 'ADR', 'Total_Nights', 'Estimated_Revenue', 'Days_In_Waiting_List']

def iqr_outlier_summary(data, col):
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((data[col] < lower) | (data[col] > upper)).sum()
    return lower, upper, n_out

for col in outlier_cols:
    lower, upper, n_out = iqr_outlier_summary(df, col)
    print(f"{col}: bounds=({lower:.2f}, {upper:.2f}) -> {n_out} outliers ({n_out/len(df)*100:.2f}%)")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), outlier_cols + ['Adults']):
    sns.boxplot(y=df[col], ax=ax, color='coral')
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 4. Data Cleaning & Preprocessing

Based on the quality checks above, we:
- Convert date columns to proper `datetime` type
- Handle missing values column by column (impute or flag as appropriate)
- Drop exact duplicate rows
- Cap extreme outliers using IQR bounds (winsorize) rather than deleting, to preserve sample size
- Standardize categorical text formatting

In [ ]:
df_clean = df.copy()

# --- 4.1 Fix data types ---
for col in ['Booking_Date', 'Arrival_Date', 'Reservation_Status_Date']:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# --- 4.2 Remove exact duplicate rows ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before - len(df_clean)} duplicate rows")

# --- 4.3 Handle missing values ---
# Company_ID: mostly missing (individual, non-company bookings) -> flag instead of impute
df_clean['Has_Company'] = df_clean['Company_ID'].notna().astype(int)
df_clean.drop(columns=['Company_ID'], inplace=True)

# Agent_ID: missing likely means direct booking with no agent -> fill with 0 (flag)
df_clean['Agent_ID'] = df_clean['Agent_ID'].fillna(0)

# Children: missing -> assume 0 children
df_clean['Children'] = df_clean['Children'].fillna(0)

# Country, Hotel_Location, Meal_Type: categorical -> fill with 'Unknown'
for col in ['Country', 'Hotel_Location', 'Meal_Type']:
    df_clean[col] = df_clean[col].fillna('Unknown')

# ADR, Satisfaction_Score: numeric, small % missing -> fill with median
for col in ['ADR', 'Satisfaction_Score']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Remaining missing values:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])


In [ ]:
# --- 4.4 Standardize categorical text ---
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

# --- 4.5 Cap outliers (winsorize) on key numeric columns ---
def cap_outliers(data, col):
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    data[col] = data[col].clip(lower=lower, upper=upper)
    return data

for col in outlier_cols:
    df_clean = cap_outliers(df_clean, col)

# --- 4.6 Remove logically invalid rows (0 guests) ---
zero_guests = (df_clean['Adults'] + df_clean['Children'] + df_clean['Babies']) == 0
df_clean = df_clean[~zero_guests]

print("Final cleaned shape:", df_clean.shape)
df_clean.head()


In [ ]:
# --- 4.7 Feature engineering for analysis ---
df_clean['Arrival_Month'] = df_clean['Arrival_Date'].dt.month_name()
df_clean['Arrival_Year'] = df_clean['Arrival_Date'].dt.year
df_clean['Total_Guests'] = df_clean['Adults'] + df_clean['Children'] + df_clean['Babies']
df_clean['Is_Canceled'] = df_clean['Is_Canceled'].astype(int)
df_clean.head()


## 5. Univariate Analysis

In [ ]:
num_cols = ['Lead_Time_Days', 'Total_Nights', 'ADR', 'Estimated_Revenue',
            'Total_Guests', 'Satisfaction_Score', 'Total_Special_Requests']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df_clean[col], kde=True, ax=ax, color='teal')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df_clean, x='Hotel_Type', ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Bookings by Hotel Type')

sns.countplot(data=df_clean, y='Market_Segment', order=df_clean['Market_Segment'].value_counts().index,
              ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Bookings by Market Segment')

sns.countplot(data=df_clean, x='Is_Canceled', ax=axes[1,0], palette='Set1')
axes[1,0].set_title('Cancellation Distribution (0=Not Canceled, 1=Canceled)')

sns.countplot(data=df_clean, y='Customer_Type', order=df_clean['Customer_Type'].value_counts().index,
              ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Bookings by Customer Type')

plt.tight_layout()
plt.show()


In [ ]:
top_countries = df_clean['Country'].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis')
plt.title('Top 10 Guest Countries by Bookings')
plt.xlabel('Number of Bookings')
plt.tight_layout()
plt.show()


## 6. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.boxplot(data=df_clean, x='Hotel_Type', y='ADR', ax=axes[0], palette='Set3')
axes[0].set_title('ADR by Hotel Type')

sns.boxplot(data=df_clean, x='Is_Canceled', y='Lead_Time_Days', ax=axes[1], palette='Set3')
axes[1].set_title('Lead Time vs Cancellation')

plt.tight_layout()
plt.show()


In [ ]:
cancel_by_segment = df_clean.groupby('Market_Segment')['Is_Canceled'].mean().sort_values(ascending=False) * 100
plt.figure(figsize=(10,5))
sns.barplot(x=cancel_by_segment.values, y=cancel_by_segment.index, palette='Reds_r')
plt.xlabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Market Segment')
plt.tight_layout()
plt.show()


In [ ]:
cancel_by_deposit = df_clean.groupby('Deposit_Type')['Is_Canceled'].mean().sort_values(ascending=False) * 100
plt.figure(figsize=(8,5))
sns.barplot(x=cancel_by_deposit.index, y=cancel_by_deposit.values, palette='Blues_r')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Deposit Type')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=df_clean.sample(min(3000, len(df_clean)), random_state=42),
                 x='Lead_Time_Days', y='ADR', hue='Is_Canceled', alpha=0.5, palette='coolwarm')
plt.title('Lead Time vs ADR (colored by Cancellation)')
plt.tight_layout()
plt.show()


In [ ]:
monthly_revenue = df_clean.groupby('Arrival_Month')['Estimated_Revenue'].sum()
month_order = ['January','February','March','April','May','June','July','August',
               'September','October','November','December']
monthly_revenue = monthly_revenue.reindex([m for m in month_order if m in monthly_revenue.index])

plt.figure(figsize=(12,5))
sns.lineplot(x=monthly_revenue.index, y=monthly_revenue.values, marker='o', color='darkgreen')
plt.xticks(rotation=45)
plt.ylabel('Total Estimated Revenue')
plt.title('Estimated Revenue by Arrival Month')
plt.tight_layout()
plt.show()


## 7. Descriptive Statistics & Group-wise Analysis

In [ ]:
summary_by_hotel = df_clean.groupby('Hotel_Type').agg(
    Bookings=('Booking_ID', 'count'),
    Avg_ADR=('ADR', 'mean'),
    Avg_Lead_Time=('Lead_Time_Days', 'mean'),
    Cancellation_Rate_Pct=('Is_Canceled', lambda x: x.mean()*100),
    Avg_Satisfaction=('Satisfaction_Score', 'mean'),
    Total_Revenue=('Estimated_Revenue', 'sum')
).round(2).sort_values('Total_Revenue', ascending=False)

summary_by_hotel


In [ ]:
summary_by_segment = df_clean.groupby('Market_Segment').agg(
    Bookings=('Booking_ID', 'count'),
    Avg_ADR=('ADR', 'mean'),
    Cancellation_Rate_Pct=('Is_Canceled', lambda x: x.mean()*100),
    Total_Revenue=('Estimated_Revenue', 'sum')
).round(2).sort_values('Total_Revenue', ascending=False)

summary_by_segment


In [ ]:
summary_by_location = df_clean.groupby('Hotel_Location').agg(
    Bookings=('Booking_ID', 'count'),
    Avg_ADR=('ADR', 'mean'),
    Cancellation_Rate_Pct=('Is_Canceled', lambda x: x.mean()*100),
    Avg_Satisfaction=('Satisfaction_Score', 'mean'),
    Total_Revenue=('Estimated_Revenue', 'sum')
).round(2).sort_values('Total_Revenue', ascending=False)

summary_by_location


In [ ]:
repeat_summary = df_clean.groupby('Is_Repeated_Guest').agg(
    Bookings=('Booking_ID', 'count'),
    Cancellation_Rate_Pct=('Is_Canceled', lambda x: x.mean()*100),
    Avg_Satisfaction=('Satisfaction_Score', 'mean'),
    Avg_ADR=('ADR', 'mean')
).round(2)
repeat_summary.index = ['New Guest', 'Repeated Guest']
repeat_summary


## 8. Correlation Analysis

In [ ]:
corr_cols = ['Lead_Time_Days', 'Total_Nights', 'ADR', 'Estimated_Revenue', 'Total_Guests',
             'Booking_Changes', 'Previous_Cancellations', 'Total_Special_Requests',
             'Satisfaction_Score', 'Is_Canceled', 'Days_In_Waiting_List']

corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Key Numeric Features')
plt.tight_layout()
plt.show()


In [ ]:
# Strongest correlations with cancellation and revenue
print("Correlation with Is_Canceled:")
print(corr_matrix['Is_Canceled'].sort_values(ascending=False))
print("\nCorrelation with Estimated_Revenue:")
print(corr_matrix['Estimated_Revenue'].sort_values(ascending=False))


## 9. Additional Visualizations

In [ ]:
plt.figure(figsize=(10,6))
sns.violinplot(data=df_clean, x='Customer_Type', y='Satisfaction_Score', palette='muted')
plt.title('Satisfaction Score Distribution by Customer Type')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
room_mismatch = (df_clean['Room_Type_Reserved'] != df_clean['Room_Type_Assigned']).mean() * 100
print(f"% of bookings where assigned room differs from reserved room: {room_mismatch:.2f}%")

mismatch_cancel = df_clean.groupby(df_clean['Room_Type_Reserved'] != df_clean['Room_Type_Assigned'])['Is_Canceled'].mean() * 100
mismatch_cancel.index = ['Room Matched', 'Room Mismatched']
plt.figure(figsize=(6,5))
sns.barplot(x=mismatch_cancel.index, y=mismatch_cancel.values, palette='Set2')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate: Room Match vs Mismatch')
plt.tight_layout()
plt.show()


In [ ]:
channel_revenue = df_clean.groupby('Distribution_Channel')['Estimated_Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(9,5))
sns.barplot(x=channel_revenue.index, y=channel_revenue.values, palette='cubehelix')
plt.ylabel('Total Estimated Revenue')
plt.title('Total Revenue by Distribution Channel')
plt.tight_layout()
plt.show()


## 10. Key Business Insights

*(Run all cells above first — the bullets below are template placeholders. Replace the `[ ]` figures with the actual numbers your run produces, e.g. from `cancel_by_segment`, `summary_by_hotel`, and `corr_matrix`.)*

1. **Cancellations are concentrated in specific market segments and deposit types.** Segments/deposit types with the highest `Cancellation_Rate_Pct` (see Section 6 & 7 outputs) represent the biggest revenue-leakage risk — long lead times combined with these segments compound the risk further, as shown by the Lead Time vs Cancellation boxplot.

2. **Lead time is a leading indicator of cancellation risk.** The correlation heatmap and boxplot in Sections 6 and 8 show bookings made far in advance are meaningfully more likely to cancel than late/near-date bookings.

3. **Hotel type and location drive materially different ADR and revenue profiles.** The `summary_by_hotel` and `summary_by_location` tables show which hotel type/location combination generates the highest average daily rate and total revenue — useful for pricing and inventory allocation decisions.

4. **Room assignment mismatches correlate with guest experience and cancellation behavior.** The mismatch-vs-cancellation comparison in Section 9 quantifies how often guests don't get their reserved room type and whether that relates to cancellation.

5. **Repeat guests behave differently from new guests.** The `repeat_summary` table shows repeat guests generally cancel less and/or report different satisfaction and ADR levels than first-time guests — a segment worth nurturing through loyalty programs.


## 11. Recommendations for Management

1. **Tighten deposit/cancellation policy for high-risk segments.** Require partial deposits or stricter cancellation terms for the market segment(s)/channels identified with the highest cancellation rates.

2. **Introduce dynamic overbooking limits based on lead time.** Since long-lead-time bookings show elevated cancellation risk, use lead-time bands to calibrate overbooking buffers by hotel/location.

3. **Prioritize room-assignment accuracy operationally.** Reduce reserved-vs-assigned room mismatches (Section 9) through better inventory/PMS controls, since mismatches are linked to a higher cancellation rate.

4. **Reallocate marketing/inventory investment toward the highest-revenue hotel type & location combinations** identified in the group-wise revenue tables, while monitoring ADR trends to avoid over-discounting high-demand properties.

5. **Launch/strengthen a loyalty program for repeat guests**, since this segment shows more favorable cancellation and/or satisfaction behavior — retaining them is cheaper than acquiring new demand.

6. **Use special requests and satisfaction score as early-warning signals.** Guests with low `Total_Special_Requests` and low `Satisfaction_Score` should be flagged for proactive service recovery before checkout/review.

7. **Build a seasonal staffing & pricing calendar** aligned to the monthly revenue trend (Section 6), ramping ADR and staffing ahead of peak-demand months and running targeted promotions in low-demand months.


---
### Appendix: Export cleaned dataset (optional)
Run the cell below if you want to save the cleaned dataset for further modeling or reporting.

In [ ]:
df_clean.to_csv('Hotel_Booking_Cleaned.csv', index=False)
print("Saved cleaned dataset: Hotel_Booking_Cleaned.csv")
